In [1]:
"""
Vergelijking Oude vs Nieuwe Kalman Parameters
==============================================
Berekent out-of-sample MAE voor beide parametersets
via dezelfde walk-forward validatie structuur (5 folds).

Oud: handmatig ingestelde parameters (symmetrisch)
Nieuw: getuunde parameters via MLE walk-forward (asymmetrisch)
"""

import numpy as np
import pandas as pd

DATA_PATH = r"C:/Users/semwi/FPL-Core-Insights/data/Kalman data/kalman_expected_goals.csv"

# ── Parameters ────────────────────────────────────────────────────────────────

# Oud model (handmatig, symmetrisch)
PARAMS_OUD = {
    "phi"          : 0.9996,
    "sigma_w_att"  : 0.0363,   # sigma_w (zelfde voor att en def)
    "sigma_w_def"  : 0.0363,
    "sigma_v"      : 0.7559,
    "delta"        : 0.15,
    "beta"         : 0.05,
    "lambda_att"   : 0.0,
    "lambda_lineup": 0.0,
    "asymmetric"   : False,
}

# Nieuw model (getuund via MLE walk-forward, asymmetrisch)
PARAMS_NIEUW = {
    "phi"          : 0.997627,
    "sigma_w_att"  : np.sqrt(0.001070),
    "sigma_w_def"  : np.sqrt(0.001253),
    "sigma_v"      : np.sqrt(0.554370),
    "delta"        : 0.243549,
    "beta"         : 0.166271,
    "lambda_att"   : 0.051143,
    "lambda_lineup": 0.190607,
    "asymmetric"   : True,
}

# ── Kalman filter functie ──────────────────────────────────────────────────────

def run_kalman_fold(df_train, df_val, params):
    """
    Train Kalman filter op df_train, evalueer MAE op df_val.
    Returns MAE home, MAE away, MAE total.
    """
    phi           = params["phi"]
    sigma_w_att   = params["sigma_w_att"]
    sigma_w_def   = params["sigma_w_def"]
    sigma_v       = params["sigma_v"]
    delta         = params["delta"]
    beta          = params["beta"]
    lambda_att    = params["lambda_att"]
    lambda_lineup = params["lambda_lineup"]
    asymmetric    = params["asymmetric"]

    teams = sorted(set(df_train["home_team"]) | set(df_train["away_team"]) |
                   set(df_val["home_team"])   | set(df_val["away_team"]))

    alpha     = {t: 0.0 for t in teams}
    gamma     = {t: 0.0 for t in teams}
    var_alpha = {t: 1.0 for t in teams}
    var_gamma = {t: 1.0 for t in teams}

    def update(row):
        h, a = row["home_team"], row["away_team"]
        ls       = row.get("lineup_strength_diff_norm", 0.0)
        pi_home  = row.get("home_pi_rating_pre", 0.0)
        pi_away  = row.get("away_pi_rating_pre", 0.0)

        a_h = phi * alpha[h]; a_a = phi * alpha[a]
        g_h = phi * gamma[h]; g_a = phi * gamma[a]
        va_h = phi**2 * var_alpha[h] + sigma_w_att**2
        va_a = phi**2 * var_alpha[a] + sigma_w_att**2
        vg_h = phi**2 * var_gamma[h] + sigma_w_def**2
        vg_a = phi**2 * var_gamma[a] + sigma_w_def**2

        xg_h_pred = delta + a_h - g_a + beta * ls
        xg_a_pred =         a_a - g_h - beta * ls

        e_h = row["home_xg"] - xg_h_pred
        e_a = row["away_xg"] - xg_a_pred

        S_h = va_h + vg_a + sigma_v**2
        S_a = va_a + vg_h + sigma_v**2

        K_ah = va_h / S_h; K_ga = vg_a / S_h
        K_aa = va_a / S_a; K_gh = vg_h / S_a

        # Gewichten
        if asymmetric:
            w_att_h = float(np.clip(1.0 + lambda_att * pi_away + lambda_lineup * (-ls), 0.1, 3.0))
            w_def_a = 1.0
            w_att_a = float(np.clip(1.0 + lambda_att * pi_home + lambda_lineup * ( ls), 0.1, 3.0))
            w_def_h = 1.0
        else:
            w_att_h = w_def_a = w_att_a = w_def_h = 1.0

        alpha[h] = a_h + K_ah * e_h * w_att_h
        alpha[a] = a_a + K_aa * e_a * w_att_a
        gamma[a] = g_a - K_ga * e_h * w_def_a
        gamma[h] = g_h - K_gh * e_a * w_def_h

        var_alpha[h] = (1 - K_ah) * va_h
        var_alpha[a] = (1 - K_aa) * va_a
        var_gamma[a] = (1 - K_ga) * vg_a
        var_gamma[h] = (1 - K_gh) * vg_h

        return xg_h_pred, xg_a_pred

    # Train: update states maar geen MAE meten
    for _, row in df_train.iterrows():
        update(row)

    # Validatie: predict EERST dan update
    errors_h, errors_a = [], []
    for _, row in df_val.iterrows():
        xg_h_pred, xg_a_pred = update(row)
        errors_h.append(abs(row["home_xg"] - xg_h_pred))
        errors_a.append(abs(row["away_xg"] - xg_a_pred))

    mae_h = np.mean(errors_h)
    mae_a = np.mean(errors_a)
    mae_t = np.mean(errors_h + errors_a)
    return mae_h, mae_a, mae_t


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    print("=" * 65)
    print("  VERGELIJKING OUD vs NIEUW KALMAN MODEL — WALK-FORWARD MAE")
    print("=" * 65)

    df = pd.read_csv(DATA_PATH, parse_dates=["date"])
    df = df.sort_values("date").reset_index(drop=True)
    df = df[df["home_xg"].notna() & df["away_xg"].notna()].copy().reset_index(drop=True)

    # Fillna voor pi en lineup kolommen (voor oud model)
    for col in ["lineup_strength_diff_norm", "home_pi_rating_pre", "away_pi_rating_pre"]:
        if col not in df.columns:
            df[col] = 0.0
        else:
            df[col] = df[col].fillna(0.0)

    seasons = sorted(df["season"].unique())
    print(f"\nData: {len(df)} wedstrijden, {len(seasons)} seizoenen")
    print(f"Seizoenen: {seasons}")

    folds = list(range(2, len(seasons)))  # min 2 train seizoenen

    results = []

    print(f"\n{'Fold':<6} {'Validatie':<12} {'MAE Oud':>9} {'MAE Nieuw':>10} {'Verschil':>9} {'Beter'}")
    print("-" * 60)

    for fold_idx, val_idx in enumerate(folds):
        train_seasons = seasons[:val_idx]
        val_season    = seasons[val_idx]

        df_train = df[df["season"].isin(train_seasons)].copy()
        df_val   = df[df["season"] == val_season].copy()

        _, _, mae_oud   = run_kalman_fold(df_train, df_val, PARAMS_OUD)
        _, _, mae_nieuw = run_kalman_fold(df_train, df_val, PARAMS_NIEUW)

        diff   = mae_nieuw - mae_oud
        beter  = "Nieuw ✅" if mae_nieuw < mae_oud else "Oud ⚠️"

        results.append({
            "fold"      : fold_idx + 1,
            "val_season": val_season,
            "mae_oud"   : mae_oud,
            "mae_nieuw" : mae_nieuw,
            "diff"      : diff,
        })

        print(f"{fold_idx+1:<6} {val_season:<12} {mae_oud:>9.4f} {mae_nieuw:>10.4f} "
              f"{diff:>+9.4f}  {beter}")

    res = pd.DataFrame(results)

    print("\n" + "=" * 65)
    print("  GEMIDDELD OVER ALLE FOLDS")
    print("=" * 65)
    print(f"  MAE Oud    : {res['mae_oud'].mean():.4f}  (std: {res['mae_oud'].std():.4f})")
    print(f"  MAE Nieuw  : {res['mae_nieuw'].mean():.4f}  (std: {res['mae_nieuw'].std():.4f})")
    print(f"  Verbetering: {res['mae_oud'].mean() - res['mae_nieuw'].mean():+.4f} "
          f"({(res['mae_oud'].mean() - res['mae_nieuw'].mean()) / res['mae_oud'].mean() * 100:+.1f}%)")
    print(f"  Naive baseline MAE: 0.7226")
    print(f"\n  Oud vs baseline : {res['mae_oud'].mean() - 0.7226:+.4f}")
    print(f"  Nieuw vs baseline: {res['mae_nieuw'].mean() - 0.7226:+.4f}")

    winner = "NIEUW model" if res['mae_nieuw'].mean() < res['mae_oud'].mean() else "OUD model"
    print(f"\n  Winnaar: {winner}")


if __name__ == "__main__":
    main()

  VERGELIJKING OUD vs NIEUW KALMAN MODEL — WALK-FORWARD MAE

Data: 2608 wedstrijden, 7 seizoenen
Seizoenen: ['2019/2020', '2020/2021', '2021/2022', '2022/2023', '2023/2024', '2024/2025', '2025/2026']

Fold   Validatie      MAE Oud  MAE Nieuw  Verschil Beter
------------------------------------------------------------
1      2021/2022       0.6128     0.6065   -0.0064  Nieuw ✅
2      2022/2023       0.6116     0.6060   -0.0056  Nieuw ✅
3      2023/2024       0.6888     0.6801   -0.0087  Nieuw ✅
4      2024/2025       0.6593     0.6549   -0.0044  Nieuw ✅
5      2025/2026       0.6231     0.6186   -0.0045  Nieuw ✅

  GEMIDDELD OVER ALLE FOLDS
  MAE Oud    : 0.6391  (std: 0.0338)
  MAE Nieuw  : 0.6332  (std: 0.0329)
  Verbetering: +0.0059 (+0.9%)
  Naive baseline MAE: 0.7226

  Oud vs baseline : -0.0835
  Nieuw vs baseline: -0.0894

  Winnaar: NIEUW model
